In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

env_path = Path.cwd() / ".env"

if not env_path.is_file():
    raise FileNotFoundError(f".env not found in: {Path.cwd()}")

load_dotenv(env_path, override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key or not api_key.strip():
    raise ValueError("OPENAI_API_KEY is missing or empty.")

client = OpenAI(api_key=api_key)

print("Python:", sys.executable)
print("Project folder:", Path.cwd())
print(".env loaded and API key found.")
print("Client created — no API request sent.")

Python: /opt/miniconda3/envs/bootcamp-env/bin/python
Project folder: /Users/mortezaarminseyfi/Desktop/AI Bootcamp/Week2_practice/day_3_practice/lab-prompt-engineering
.env loaded and API key found.
Client created — no API request sent.


In [7]:
MODEL = "gpt-4o-mini"
TEMPERATURE = 0.7

# Keep actual responses and token usage for later analysis.
call_history = []

def call_openai(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=TEMPERATURE,
        max_completion_tokens=500,
    )

    result = response.choices[0].message.content or ""

    call_history.append({
        "prompt": prompt,
        "output": result,
        "model": response.model,
        "temperature": TEMPERATURE,
        "finish_reason": response.choices[0].finish_reason,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
    })

    return result

print("Helper ready — no API request sent.")

Helper ready — no API request sent.


In [3]:
sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""

product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""

extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

print("Three v1 prompts defined.")

Three v1 prompts defined.


In [8]:
sentiment_result_v1 = call_openai(sentiment_prompt_v1)

print("Sentiment Analysis Result:")
print(sentiment_result_v1)

print("\nToken usage:")
print("Input:", call_history[-1]["input_tokens"])
print("Output:", call_history[-1]["output_tokens"])

Sentiment Analysis Result:
The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Token usage:
Input: 26
Output: 17


In [9]:
product_result_v1 = call_openai(product_prompt_v1)

print("=== Product Description Result ===")
print(product_result_v1)

extraction_result_v1 = call_openai(extraction_prompt_v1)

print("\n=== Data Extraction Result ===")
print(extraction_result_v1)

=== Product Description Result ===
**Product Name: SleekWave Wireless Mouse**

**Price: $29.99**

**Product Description:**

Elevate your computing experience with the SleekWave Wireless Mouse, designed for comfort, style, and seamless functionality. Priced at just $29.99, this versatile mouse is perfect for both professionals and casual users alike.

**Key Features:**

- **Ergonomic Design:** The SleekWave is crafted to fit comfortably in your hand, reducing strain during long hours of use. Its sleek contours allow for a natural grip, making it ideal for extended work sessions or gaming marathons.

- **Wireless Freedom:** Enjoy the convenience of wireless connectivity with a reliable 2.4GHz connection. Say goodbye to tangled cords and limited mobility—navigate your workspace with ease and precision up to 33 feet away.

- **Long Battery Life:** Equipped with an energy-efficient battery, the SleekWave ensures you stay productive without frequent interruptions. With up to 12 months of use

### Initial v1 observations

- Sentiment: The response correctly identifies positive sentiment, but
  returns two alternative labels and Markdown rather than one standard label.
- Product description: The response invents a product name and unsupported
  specifications, including connectivity, range, battery life, and compatibility.
- Data extraction: The response captures several explicit details without
  inventing a year, but incorrectly interprets an item number as an order number.
  The output is a prose list rather than a defined machine-readable format.

These are observations from one run per task, not consistency estimates.

In [10]:
from concurrent.futures import ThreadPoolExecutor

def run_prompt_tests(prompt, runs=5):
    with ThreadPoolExecutor(max_workers=3) as executor:
        outputs = list(
            executor.map(call_openai, [prompt] * runs)
        )

    for number, output in enumerate(outputs, start=1):
        print(f"\n--- Run {number} ---")
        print(output)

    return outputs

print("Repeated-testing helper ready — no API request sent.")

Repeated-testing helper ready — no API request sent.


In [11]:
sentiment_v1_runs_5 = run_prompt_tests(
    sentiment_prompt_v1,
    runs=5
)


--- Run 1 ---
The customer message can be classified as positive feedback or a positive review.

--- Run 2 ---
This customer message can be classified as **positive feedback** or **satisfaction**.

--- Run 3 ---
The customer message can be classified as positive feedback or a positive review.

--- Run 4 ---
This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

--- Run 5 ---
The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."


In [12]:
def evaluate_sentiment(outputs, expected_label):
    allowed_labels = {"positive", "negative", "neutral"}

    # Remove surrounding whitespace, but keep case and punctuation.
    cleaned = [output.strip() for output in outputs]

    format_passes = sum(
        output in allowed_labels for output in cleaned
    )

    correct_label_passes = sum(
        output == expected_label for output in cleaned
    )

    return {
        "runs": len(outputs),
        "format_compliance_pct": 100 * format_passes / len(outputs),
        "correct_standard_label_pct": (
            100 * correct_label_passes / len(outputs)
        ),
        "unique_texts": len(set(cleaned)),
    }

sentiment_v1_metrics_5 = evaluate_sentiment(
    sentiment_v1_runs_5,
    expected_label="positive",
)

print(sentiment_v1_metrics_5)

{'runs': 5, 'format_compliance_pct': 0.0, 'correct_standard_label_pct': 0.0, 'unique_texts': 4}


In [13]:
product_v1_runs_5 = run_prompt_tests(
    product_prompt_v1,
    runs=5,
)


--- Run 1 ---
**Product Description: Wireless Precision Mouse**

Elevate your computing experience with our Wireless Precision Mouse, designed for seamless connectivity and effortless navigation. Priced at just $29.99, this sleek and stylish mouse combines ergonomic comfort with cutting-edge technology, making it the perfect companion for both work and play.

**Key Features:**

- **Wireless Freedom:** Enjoy the convenience of a clutter-free workspace with our advanced 2.4GHz wireless technology. The included USB receiver ensures a stable connection with a range of up to 33 feet, allowing you to move freely without any tangles.

- **Ergonomic Design:** Crafted with comfort in mind, the Wireless Precision Mouse features a contoured shape that fits snugly in your hand, reducing strain during long hours of use. Its textured grip provides added stability, ensuring you stay in control.

- **High Precision Tracking:** Equipped with a high-precision optical sensor, this mouse delivers smooth 

### Product v1 — five-run analysis

All five responses contain unsupported product specifications.
Examples include battery life, battery type, programmable buttons,
DPI settings, and operating-system compatibility.

Unsupported-claim-free rate: 0/5 = 0%.

The wording varies, but the main failure is factual grounding,
not textual variation.

In [14]:
extraction_v1_runs_5 = run_prompt_tests(
    extraction_prompt_v1,
    runs=5,
)


--- Run 1 ---
Here is the extracted information from the customer feedback:

- **Order Number:** #12345
- **Order Date:** March 15th
- **Delivery Speed:** Fast
- **Packaging Condition:** Damaged

--- Run 2 ---
- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Issue**: Damaged packaging

--- Run 3 ---
Here is the extracted information from the customer feedback:

- Order Number: #12345
- Order Date: March 15th
- Delivery Speed: Fast
- Packaging Condition: Damaged

--- Run 4 ---
Here are the key details extracted from the customer feedback:

- **Order Number:** #12345
- **Order Date:** March 15th
- **Delivery Speed:** Fast
- **Packaging Condition:** Damaged

--- Run 5 ---
Here is the extracted information from the customer feedback:

- **Order Number:** #12345
- **Order Date:** March 15th
- **Delivery Speed:** Fast
- **Packaging Condition:** Damaged


### Data extraction v1 — five-run analysis

All five responses incorrectly label the item number as an order number.
All preserve the stated date without inventing a year and capture
fast delivery and damaged packaging.

Field names and Markdown formatting vary across runs.
None of the responses is directly parseable as JSON.

- Correct identifier type: 0/5 = 0%.
- Date preserved without an invented year: 5/5 = 100%.
- Delivery and packaging details captured: 5/5 = 100%.
- JSON parseability: 0/5 = 0%.

JSON is our target output contract; the v1 prompt does not yet request it.

In [15]:
sentiment_v1_runs_10 = run_prompt_tests(
    sentiment_prompt_v1,
    runs=10,
)

sentiment_v1_metrics_10 = evaluate_sentiment(
    sentiment_v1_runs_10,
    expected_label="positive",
)

print("\n=== Metrics ===")
print(sentiment_v1_metrics_10)


--- Run 1 ---
This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

--- Run 2 ---
This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

--- Run 3 ---
The customer message can be classified as **Positive Feedback**.

--- Run 4 ---
This customer message can be classified as **positive feedback** or **customer satisfaction**.

--- Run 5 ---
This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

--- Run 6 ---
This customer message can be classified as **positive feedback** or **satisfaction**.

--- Run 7 ---
The customer message can be classified as **Positive Feedback**.

--- Run 8 ---
This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

--- Run 9 ---
The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

--- Run 10 ---
This customer message can be classified as **Positive Feedback** or **Cu

In [16]:
print("=== Product v1: 10 runs ===")

product_v1_runs_10 = run_prompt_tests(
    product_prompt_v1,
    runs=10,
)

print("\n=== Extraction v1: 10 runs ===")

extraction_v1_runs_10 = run_prompt_tests(
    extraction_prompt_v1,
    runs=10,
)

=== Product v1: 10 runs ===

--- Run 1 ---
**Product Description: Wireless Freedom Mouse**

Elevate your computing experience with our Wireless Freedom Mouse, the perfect blend of style, performance, and convenience—all for just $29.99! Designed for both work and play, this sleek, ergonomic mouse provides the ultimate comfort for extended use, ensuring you stay focused and productive throughout the day.

**Key Features:**

- **Seamless Connectivity:** Enjoy the freedom of wireless technology with a reliable 2.4 GHz connection. Simply plug in the USB receiver, and you’re ready to go—no drivers needed!

- **Ergonomic Design:** Crafted with your comfort in mind, the Wireless Freedom Mouse features a contoured shape that fits naturally in your hand, reducing strain and fatigue during long hours of use.

- **Precision Tracking:** With an adjustable DPI (up to 2400), this mouse offers smooth and accurate tracking on a variety of surfaces, making it perfect for gaming, graphic design, or ever

### v1 — ten-run analysis

Product:
- All 10 responses contain unsupported specifications.
- Unsupported-claim-free rate: 0/10 = 0%.
- Claimed specifications vary, including battery type and battery life.

Extraction:
- Only run 7 correctly labels 12345 as an item number.
- Correct identifier type: 1/10 = 10%.
- All 10 preserve the date without inventing a year and capture
  fast delivery and damaged packaging.
- JSON parseability: 0/10 = 0%.
- Field names vary across responses.

More runs exposed an occasional correct identifier interpretation,
but the dominant semantic error remains.

In [17]:
print("=== Sentiment v1: 15 runs ===")
sentiment_v1_runs_15 = run_prompt_tests(
    sentiment_prompt_v1, runs=15
)

print("\n=== Sentiment metrics ===")
print(evaluate_sentiment(
    sentiment_v1_runs_15,
    expected_label="positive",
))

print("\n=== Product v1: 15 runs ===")
product_v1_runs_15 = run_prompt_tests(
    product_prompt_v1, runs=15
)

print("\n=== Extraction v1: 15 runs ===")
extraction_v1_runs_15 = run_prompt_tests(
    extraction_prompt_v1, runs=15
)

=== Sentiment v1: 15 runs ===

--- Run 1 ---
This customer message can be classified as **Positive Feedback**.

--- Run 2 ---
The customer message can be classified as "Positive Feedback" or "Satisfaction."

--- Run 3 ---
This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

--- Run 4 ---
The customer message can be classified as **Positive Feedback**.

--- Run 5 ---
The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

--- Run 6 ---
The customer message can be classified as positive feedback or a customer compliment.

--- Run 7 ---
This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

--- Run 8 ---
The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

--- Run 9 ---
The customer message can be classified as **Positive Feedback** or **Praise**.

--- Run 10 ---
The customer message can be classified as **positive feedback** or **custome

### Baseline failure analysis — v1

| Metric | 5 runs | 10 runs | 15 runs |
|---|---:|---:|---:|
| Sentiment: exact expected label | 0% | 0% | 0% |
| Product: no unsupported claims | 0% | 0% | 0% |
| Extraction: correct item identifier meaning | 0% | 10% | 26.7% |
| Extraction: valid JSON | 0% | 0% | 0% |

Manual review of the 15-run tests:
- Sentiment: all responses correctly convey positive sentiment,
  but none follows the target single-label output contract.
- Product: all responses contain unsupported specifications.
- Extraction: runs 4, 8, 10, and 13 preserve the item identifier meaning.
  The other 11 incorrectly label it as an order number.
- All extraction responses preserve the date without inventing a year
  and capture fast delivery and damaged packaging.

These metrics measure different requirements and are reported separately.
Output contracts were evaluation targets, not explicit v1 instructions.
Repeated tests on one input do not establish performance on unseen inputs.

In [18]:
sentiment_prompt_v2 = """
Classify the sentiment of the customer message.

Allowed labels:
- positive: expresses satisfaction or praise
- negative: expresses dissatisfaction or criticism
- neutral: expresses no clear positive or negative sentiment

Return exactly one lowercase word:
positive, negative, or neutral.

Do not include explanations, punctuation, quotation marks, or Markdown.

Customer message:
"I love this product! It's exactly what I needed."
"""

In [19]:
sentiment_v2_runs_5 = run_prompt_tests(
    sentiment_prompt_v2,
    runs=5,
)

sentiment_v2_metrics_5 = evaluate_sentiment(
    sentiment_v2_runs_5,
    expected_label="positive",
)

print("\n=== v2 metrics ===")
print(sentiment_v2_metrics_5)


--- Run 1 ---
positive

--- Run 2 ---
positive

--- Run 3 ---
positive

--- Run 4 ---
positive

--- Run 5 ---
positive

=== v2 metrics ===
{'runs': 5, 'format_compliance_pct': 100.0, 'correct_standard_label_pct': 100.0, 'unique_texts': 1}


### Sentiment v2 — five-run evaluation

All five responses return exactly "positive".
Format compliance and correct standard-label rate improved
from 0% in v1 to 100% in v2 on the same input.

Explicit label definitions and output constraints resolved the
observed formatting failure without few-shot examples.
Generalization to other messages has not yet been tested.

In [20]:
product_prompt_v2 = """
Write a concise product description using only the facts below.

Provided facts:
- Product: wireless mouse
- Price: $29.99

Requirements:
- First line must be exactly: Wireless Mouse
- Then write one paragraph of 30–50 words.
- Include the price exactly as $29.99.
- Use a clear, friendly, professional tone.
- Do not use Markdown, bullet points, or additional sections.
- Do not invent a brand or model name.
- Do not add unsupported specifications or performance claims.
- Do not claim battery life, battery type, connection protocol,
  range, compatibility, ergonomics, sensor type, programmable
  buttons, included accessories, colors, or setup requirements.
- If information is missing, omit it.
- You may describe freedom from a mouse cable as a direct
  implication of the provided wireless feature.
"""

In [21]:
product_v2_runs_5 = run_prompt_tests(
    product_prompt_v2,
    runs=5,
)

print("\n=== Paragraph word counts ===")
for number, output in enumerate(product_v2_runs_5, start=1):
    lines = output.strip().splitlines()
    paragraph = " ".join(lines[1:]).strip()
    print(f"Run {number}: {len(paragraph.split())} words")


--- Run 1 ---
Wireless Mouse  
Experience the convenience of a wireless mouse designed to enhance your computing experience. Enjoy the freedom of movement without the hassle of cables, making it perfect for home or office use. Available now for just $29.99.

--- Run 2 ---
Wireless Mouse  
Experience the convenience of a wireless mouse that offers freedom from tangled cables. Designed for ease of use, this mouse is perfect for both work and play. Enjoy smooth navigation and enhance your productivity for just $29.99.

--- Run 3 ---
Wireless Mouse  
Experience the convenience of a wireless mouse that provides you with the freedom to move without the hassle of cables. Priced at just $29.99, this essential accessory is perfect for enhancing your productivity and comfort while working or gaming.

--- Run 4 ---
Wireless Mouse  
Experience the convenience of a wireless design with our reliable mouse. Priced at $29.99, it offers the freedom to navigate without the hassle of a cable, making it 

In [22]:
product_prompt_v2b = product_prompt_v2 + """

Additional grounding rules:
- Do not claim reliability, smooth or seamless navigation,
  precision, comfort, ease of use, or increased productivity.
- Do not claim suitability for gaming, travel, or particular users.
- Avoid evaluative claims such as "perfect", "ideal", or "essential".
- Limit the description to the product type, wireless operation,
  freedom from a mouse cable, and the stated price.
- Use a restrained, informative tone rather than promotional praise.
"""

In [23]:
product_v2b_runs_5 = run_prompt_tests(
    product_prompt_v2b,
    runs=5,
)

print("\n=== Paragraph word counts ===")
for number, output in enumerate(product_v2b_runs_5, start=1):
    lines = output.strip().splitlines()
    paragraph = " ".join(lines[1:]).strip()
    print(f"Run {number}: {len(paragraph.split())} words")


--- Run 1 ---
Wireless Mouse  
Experience the convenience of a wireless mouse, allowing for unrestricted movement without the hassle of a cable. Priced at $29.99, this device provides an efficient solution for your computing needs, making it easier to navigate your digital world.

--- Run 2 ---
Wireless Mouse  
This wireless mouse offers the convenience of cable-free operation, allowing for greater freedom of movement on your workspace. Enjoy a tidy setup without the hassle of cords. Priced at $29.99, it provides a straightforward solution for your computing needs.

--- Run 3 ---
Wireless Mouse  
Experience the convenience of a wireless mouse, allowing for greater flexibility without the constraints of a cable. Priced at $29.99, this device provides a practical solution for your computing needs. Enjoy the freedom of movement while navigating your computer tasks.

--- Run 4 ---
Wireless Mouse  
Experience the convenience of a wireless mouse, designed to provide the freedom of movement 

In [24]:
extraction_prompt_v2 = """
Extract information from the customer feedback below.

Return only a valid JSON object with exactly these keys:
- "item_id": string or null
- "order_date": string or null
- "delivery_speed": string or null
- "packaging_condition": string or null

Rules:
- Extract only information explicitly stated in the feedback.
- An item identifier is not an order identifier.
- Remove a leading # from the item identifier.
- Preserve the date wording exactly; do not invent a year.
- Use lowercase for delivery speed and packaging condition.
- Use null for missing information.
- Do not include Markdown, explanations, or additional keys.

Customer feedback:
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

In [25]:
extraction_v2_runs_5 = run_prompt_tests(
    extraction_prompt_v2,
    runs=5,
)


--- Run 1 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 2 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 3 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 4 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 5 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}


In [26]:
import json

expected_extraction = {
    "item_id": "12345",
    "order_date": "March 15th",
    "delivery_speed": "fast",
    "packaging_condition": "damaged",
}

def evaluate_extraction(outputs, expected):
    results = []

    for run, output in enumerate(outputs, start=1):
        try:
            parsed = json.loads(output)
            valid_json = True
        except json.JSONDecodeError:
            parsed = None
            valid_json = False

        correct_schema = (
            isinstance(parsed, dict)
            and set(parsed) == set(expected)
            and all(
                value is None or isinstance(value, str)
                for value in parsed.values()
            )
        )

        correct_values = correct_schema and parsed == expected

        results.append({
            "run": run,
            "valid_json": valid_json,
            "correct_schema": correct_schema,
            "correct_values": correct_values,
        })

    for result in results:
        print(result)

    print("\n=== Summary ===")
    for metric in ("valid_json", "correct_schema", "correct_values"):
        passed = sum(result[metric] for result in results)
        print(
            f"{metric}: {passed}/{len(results)} "
            f"= {100 * passed / len(results):.1f}%"
        )

    return results

extraction_v2_checks_5 = evaluate_extraction(
    extraction_v2_runs_5,
    expected_extraction,
)

{'run': 1, 'valid_json': True, 'correct_schema': True, 'correct_values': True}
{'run': 2, 'valid_json': True, 'correct_schema': True, 'correct_values': True}
{'run': 3, 'valid_json': True, 'correct_schema': True, 'correct_values': True}
{'run': 4, 'valid_json': True, 'correct_schema': True, 'correct_values': True}
{'run': 5, 'valid_json': True, 'correct_schema': True, 'correct_values': True}

=== Summary ===
valid_json: 5/5 = 100.0%
correct_schema: 5/5 = 100.0%
correct_values: 5/5 = 100.0%


In [27]:
sentiment_prompt_v3 = """
Classify the sentiment of the customer message.

Allowed labels:
- positive: expresses satisfaction or praise
- negative: expresses dissatisfaction or criticism
- neutral: factual, or mixed feedback with no dominant sentiment

For mixed feedback, choose the dominant sentiment.
If neither positive nor negative sentiment dominates, choose neutral.

Return exactly one lowercase word:
positive, negative, or neutral.
Do not include explanations, punctuation, quotation marks, or Markdown.

Examples:

Message: "The delivery was quick and the product works perfectly."
Label: positive

Message: "The item arrived broken and support did not help."
Label: negative

Message: "Delivery was quick, but the packaging was damaged."
Label: neutral

Customer message:
"I love this product! It's exactly what I needed."
"""

In [28]:
sentiment_v3_runs_15 = run_prompt_tests(
    sentiment_prompt_v3,
    runs=15,
)

sentiment_v3_metrics_15 = evaluate_sentiment(
    sentiment_v3_runs_15,
    expected_label="positive",
)

print("\n=== Sentiment v3 metrics ===")
print(sentiment_v3_metrics_15)


--- Run 1 ---
positive

--- Run 2 ---
positive

--- Run 3 ---
positive

--- Run 4 ---
positive

--- Run 5 ---
positive

--- Run 6 ---
positive

--- Run 7 ---
positive

--- Run 8 ---
positive

--- Run 9 ---
positive

--- Run 10 ---
positive

--- Run 11 ---
positive

--- Run 12 ---
positive

--- Run 13 ---
positive

--- Run 14 ---
positive

--- Run 15 ---
positive

=== Sentiment v3 metrics ===
{'runs': 15, 'format_compliance_pct': 100.0, 'correct_standard_label_pct': 100.0, 'unique_texts': 1}


In [29]:
extraction_prompt_v3 = """
Extract structured information from customer feedback.

Before producing the final answer, work through these checks internally:
1. Locate the identifier and determine whether it refers to an item
   or an order. Do not treat an order number as an item identifier.
2. Locate the order date. Preserve its original wording.
   Do not infer a missing year.
3. Identify the delivery speed and packaging condition separately.
4. Check that every extracted value is supported by the feedback.
   Use null for missing information.
5. Verify the required JSON keys and value types.

Return only a valid JSON object with exactly these keys:
- "item_id": string or null; remove a leading #
- "order_date": string or null
- "delivery_speed": lowercase string or null
- "packaging_condition": lowercase string or null

Do not include reasoning, explanations, Markdown, or extra keys.

Customer feedback:
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

In [30]:
extraction_v3_runs_15 = run_prompt_tests(
    extraction_prompt_v3,
    runs=15,
)

extraction_v3_checks_15 = evaluate_extraction(
    extraction_v3_runs_15,
    expected_extraction,
)


--- Run 1 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 2 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 3 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 4 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 5 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 6 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 7 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 8 ---
{
  "item_id": "12345",
  "order_date": "March 

In [31]:
product_prompt_v3 = """
Write a product description using only the provided facts.

Output requirements:
- First line: the product type exactly as provided.
- Then one paragraph of 30–50 words, excluding the title.
- Include the provided price exactly.
- Use a restrained, clear, professional tone.
- No Markdown, bullets, or additional sections.

Grounding rules:
- Do not invent brands, model names, specifications, accessories,
  compatibility, or performance benefits.
- Do not claim reliability, comfort, precision, ease of use,
  improved productivity, or suitability for particular users.
- For a wireless product, cable-free operation is an allowed implication.
  Do not imply unlimited range or unrestricted movement.
- Omit missing details.

Examples:

Input:
Product: Wired Keyboard
Price: $19.99

Output:
Wired Keyboard
This wired keyboard is priced at $19.99. Its wired connection is the only connection detail provided. Consider this option if you are looking for a keyboard, and consult the product specifications for any additional details.

Input:
Product: Wireless Headphones
Price: $49.99

Output:
Wireless Headphones
These wireless headphones are available for $49.99. Wireless operation means there is no audio cable connecting the headphones to the source device during use. Consult the product specifications for details beyond the stated product type and price.

Now write the description for:
Product: Wireless Mouse
Price: $29.99
"""

In [32]:
product_v3_runs_15 = run_prompt_tests(
    product_prompt_v3,
    runs=15,
)

print("\n=== Structure checks ===")

for number, output in enumerate(product_v3_runs_15, start=1):
    lines = [
        line.strip()
        for line in output.strip().splitlines()
        if line.strip()
    ]

    title_ok = bool(lines) and lines[0] == "Wireless Mouse"
    paragraph_ok = len(lines) == 2
    paragraph = " ".join(lines[1:])
    word_count = len(paragraph.split())
    length_ok = 30 <= word_count <= 50
    price_ok = "$29.99" in paragraph

    print({
        "run": number,
        "title_ok": title_ok,
        "paragraph_ok": paragraph_ok,
        "words": word_count,
        "length_ok": length_ok,
        "price_ok": price_ok,
    })


--- Run 1 ---
Wireless Mouse  
This wireless mouse is available for $29.99. Its wireless operation eliminates the need for a cable connection to the source device. For further information, please refer to the product specifications for additional details beyond the type and price mentioned.

--- Run 2 ---
Wireless Mouse  
This wireless mouse is priced at $29.99. The absence of a cable allows for a more flexible use experience. For further information regarding specifications and features, please refer to the product details.

--- Run 3 ---
Wireless Mouse  
This wireless mouse is priced at $29.99. It operates without a cable connecting it to the source device, allowing for greater flexibility during use. For further information, please refer to the product specifications.

--- Run 4 ---
Wireless Mouse  
This wireless mouse is available for $29.99. It features a cable-free design, allowing for convenient use without the limitations of a wired connection. For further information on its s

### Product v3 — fifteen-run evaluation

All 15 outputs pass title, paragraph structure, word-count,
and price checks.

Manual grounding review:
- 12/15 outputs pass: 80%.
- Runs 4, 8, and 13 contain unsupported convenience claims
  or overly broad statements about removing wired limitations.
- Cable-related flexibility is accepted as an allowed implication
  of wireless operation.

Few-shot examples removed the observed fabricated technical
specifications, but some promotional overstatement remains.
The descriptions are more grounded but less compelling.
The strict >80% target has not yet been reached.

In [33]:
sentiment_variations = [
    {
        "message": "It stopped working after two days. I want a refund.",
        "expected": "negative",
    },
    {
        "message": "The package arrived on Tuesday.",
        "expected": "neutral",
    },
    {
        "message": "Setup took a while, but overall I am very happy with it.",
        "expected": "positive",
    },
    {
        "message": "Fantastic—another broken item and no reply from support.",
        "expected": "negative",
    },
]

In [34]:
sentiment_variation_results = []

for case in sentiment_variations:
    prompt = sentiment_prompt_v3.replace(
        "I love this product! It's exactly what I needed.",
        case["message"],
    )

    output = call_openai(prompt).strip()

    result = {
        "message": case["message"],
        "expected": case["expected"],
        "actual": output,
        "passed": output == case["expected"],
    }

    sentiment_variation_results.append(result)
    print(result)

{'message': 'It stopped working after two days. I want a refund.', 'expected': 'negative', 'actual': 'negative', 'passed': True}
{'message': 'The package arrived on Tuesday.', 'expected': 'neutral', 'actual': 'neutral', 'passed': True}
{'message': 'Setup took a while, but overall I am very happy with it.', 'expected': 'positive', 'actual': 'positive', 'passed': True}
{'message': 'Fantastic—another broken item and no reply from support.', 'expected': 'negative', 'actual': 'negative', 'passed': True}


In [35]:
extraction_variations = [
    {
        "feedback": "The packaging was damaged.",
        "expected": {
            "item_id": None,
            "order_date": None,
            "delivery_speed": None,
            "packaging_condition": "damaged",
        },
    },
    {
        "feedback": (
            "Order #7788 was placed on April 2nd. "
            "Delivery was slow."
        ),
        "expected": {
            "item_id": None,
            "order_date": "April 2nd",
            "delivery_speed": "slow",
            "packaging_condition": None,
        },
    },
    {
        "feedback": (
            "I ordered item #54321 under order #98765 on June 8th. "
            "Delivery was fast and the packaging was intact."
        ),
        "expected": {
            "item_id": "54321",
            "order_date": "June 8th",
            "delivery_speed": "fast",
            "packaging_condition": "intact",
        },
    },
]

In [36]:
original_feedback = (
    "I ordered item #12345 on March 15th. "
    "The delivery was fast but the packaging was damaged."
)

extraction_variation_results = []

for number, case in enumerate(extraction_variations, start=1):
    prompt = extraction_prompt_v3.replace(
        original_feedback,
        case["feedback"],
    )

    output = call_openai(prompt)

    print(f"\n=== Variation {number} ===")
    print(output)

    checks = evaluate_extraction(
        [output],
        case["expected"],
    )

    extraction_variation_results.append({
        "feedback": case["feedback"],
        "expected": case["expected"],
        "output": output,
        "checks": checks,
    })


=== Variation 1 ===
{
  "item_id": null,
  "order_date": null,
  "delivery_speed": null,
  "packaging_condition": "damaged"
}
{'run': 1, 'valid_json': True, 'correct_schema': True, 'correct_values': True}

=== Summary ===
valid_json: 1/1 = 100.0%
correct_schema: 1/1 = 100.0%
correct_values: 1/1 = 100.0%

=== Variation 2 ===
{
  "item_id": null,
  "order_date": "April 2nd",
  "delivery_speed": "slow",
  "packaging_condition": null
}
{'run': 1, 'valid_json': True, 'correct_schema': True, 'correct_values': True}

=== Summary ===
valid_json: 1/1 = 100.0%
correct_schema: 1/1 = 100.0%
correct_values: 1/1 = 100.0%

=== Variation 3 ===
{
  "item_id": "54321",
  "order_date": "June 8th",
  "delivery_speed": "fast",
  "packaging_condition": "intact"
}
{'run': 1, 'valid_json': True, 'correct_schema': True, 'correct_values': True}

=== Summary ===
valid_json: 1/1 = 100.0%
correct_schema: 1/1 = 100.0%
correct_values: 1/1 = 100.0%


In [37]:
product_variations = [
    {"product": "Wired Mouse", "price": "$14.99"},
    {"product": "Wireless Keyboard", "price": "$39.99"},
    {"product": "Notebook", "price": "$4.99"},
]

In [38]:
product_variation_results = []

original_product_input = (
    "Now write the description for:\n"
    "Product: Wireless Mouse\n"
    "Price: $29.99"
)

for case in product_variations:
    new_product_input = (
        "Now write the description for:\n"
        f"Product: {case['product']}\n"
        f"Price: {case['price']}"
    )

    prompt = product_prompt_v3.replace(
        original_product_input,
        new_product_input,
    )

    # Confirm that the intended input was actually replaced.
    assert prompt != product_prompt_v3

    output = call_openai(prompt)

    lines = [
        line.strip()
        for line in output.strip().splitlines()
        if line.strip()
    ]
    paragraph = " ".join(lines[1:])
    word_count = len(paragraph.split())

    checks = {
        "title_ok": bool(lines) and lines[0] == case["product"],
        "paragraph_ok": len(lines) == 2,
        "words": word_count,
        "length_ok": 30 <= word_count <= 50,
        "price_ok": case["price"] in paragraph,
    }

    product_variation_results.append({
        **case,
        "output": output,
        "checks": checks,
    })

    print(f"\n=== {case['product']} ===")
    print(output)
    print(checks)


=== Wired Mouse ===
Wired Mouse  
This wired mouse is priced at $14.99. It features a wired connection, ensuring consistent performance during use. For further details, please refer to the product specifications.
{'title_ok': True, 'paragraph_ok': True, 'words': 26, 'length_ok': False, 'price_ok': True}

=== Wireless Keyboard ===
Wireless Keyboard  
This wireless keyboard is available for $39.99. Its wireless operation eliminates the need for a cable connection to the source device. For further information, please refer to the product specifications.
{'title_ok': True, 'paragraph_ok': True, 'words': 30, 'length_ok': True, 'price_ok': True}

=== Notebook ===
Notebook  
This notebook is priced at $4.99. It provides a simple solution for note-taking and writing tasks. For further details regarding size, paper type, or other specifications, please refer to the product information.
{'title_ok': True, 'paragraph_ok': True, 'words': 32, 'length_ok': True, 'price_ok': True}


In [39]:
product_prompt_v3_final = product_prompt_v3 + """

Final checks before answering:
- Keep the paragraph between 30 and 50 words.
- Mentioning a wired connection does not justify claims about
  consistent performance, reliability, speed, or responsiveness.
- Do not claim convenience, a streamlined experience,
  or freedom from all limitations.
- Cable-free operation only means no connection cable is required
  during wireless operation; it does not mean unrestricted movement.
- If the product type is ambiguous, preserve the provided name
  without assuming its category, materials, features, or intended use.
- You may advise checking further product information, but do not
  imply that a particular specification sheet is available.
- Recheck every factual claim against the provided facts.
"""

In [40]:
prompt = product_prompt_v3_final.replace(
    original_product_input,
    new_product_input,
)

assert prompt != product_prompt_v3_final

In [41]:
product_final_variation_results = []

In [42]:
product_final_variation_results.append({
    **case,
    "output": output,
    "checks": checks,
})

In [43]:
product_final_variation_results = []

for case in product_variations:
    new_product_input = (
        "Now write the description for:\n"
        f"Product: {case['product']}\n"
        f"Price: {case['price']}"
    )

    prompt = product_prompt_v3_final.replace(
        original_product_input,
        new_product_input,
    )

    assert prompt != product_prompt_v3_final

    output = call_openai(prompt)

    lines = [
        line.strip()
        for line in output.strip().splitlines()
        if line.strip()
    ]
    paragraph = " ".join(lines[1:])
    word_count = len(paragraph.split())

    checks = {
        "title_ok": bool(lines) and lines[0] == case["product"],
        "paragraph_ok": len(lines) == 2,
        "words": word_count,
        "length_ok": 30 <= word_count <= 50,
        "price_ok": case["price"] in paragraph,
    }

    product_final_variation_results.append({
        **case,
        "output": output,
        "checks": checks,
    })

    print(f"\n=== {case['product']} ===")
    print(output)
    print(checks)


=== Wired Mouse ===
Wired Mouse  
This wired mouse is priced at $14.99. It features a wired connection, which is the only detail provided. For further information about its specifications or features, please refer to additional product resources.
{'title_ok': True, 'paragraph_ok': True, 'words': 32, 'length_ok': True, 'price_ok': True}

=== Wireless Keyboard ===
Wireless Keyboard  
This wireless keyboard is offered at a price of $39.99. The absence of a connection cable allows for a cable-free operation during use. For further information about features and specifications, please consult the product details.
{'title_ok': True, 'paragraph_ok': True, 'words': 35, 'length_ok': True, 'price_ok': True}

=== Notebook ===
Notebook  
This notebook is available for $4.99. It serves as a versatile option for various writing tasks. For further details on its specifications and features, please consult additional product information.
{'title_ok': True, 'paragraph_ok': True, 'words': 29, 'length_o

In [44]:
product_prompt_v3_final_b = product_prompt_v3_final.replace(
    "Now write the description for:",
    """Input:
Product: Pad
Price: $9.99

Output:
Pad
This product is listed as Pad and priced at $9.99. The supplied information does not specify its category or intended use. Before purchasing, check further product information to determine whether it meets your requirements.

Now write the description for:""",
)

In [45]:
notebook_test_prompt = product_prompt_v3_final_b.replace(
    original_product_input,
    "Now write the description for:\n"
    "Product: Notebook\n"
    "Price: $4.99",
)

assert notebook_test_prompt != product_prompt_v3_final_b

notebook_final_output = call_openai(notebook_test_prompt)

lines = [
    line.strip()
    for line in notebook_final_output.strip().splitlines()
    if line.strip()
]

print(notebook_final_output)
print("\nParagraph words:", len(" ".join(lines[1:]).split()))

Notebook  
This product is identified as a Notebook and is priced at $4.99. The details provided do not specify its features or intended use. It is advisable to review additional information to ascertain whether this notebook meets your needs.

Paragraph words: 38


In [46]:
product_final_runs_15 = run_prompt_tests(
    product_prompt_v3_final_b,
    runs=15,
)

product_final_structure_checks = []

for number, output in enumerate(product_final_runs_15, start=1):
    lines = [
        line.strip()
        for line in output.strip().splitlines()
        if line.strip()
    ]

    paragraph = " ".join(lines[1:])
    word_count = len(paragraph.split())

    checks = {
        "run": number,
        "title_ok": bool(lines) and lines[0] == "Wireless Mouse",
        "paragraph_ok": len(lines) == 2,
        "words": word_count,
        "length_ok": 30 <= word_count <= 50,
        "price_ok": "$29.99" in paragraph,
        "no_markdown": not any(
            marker in output for marker in ("**", "```", "##")
        ),
    }

    checks["structure_pass"] = all(
        checks[key]
        for key in (
            "title_ok",
            "paragraph_ok",
            "length_ok",
            "price_ok",
            "no_markdown",
        )
    )

    product_final_structure_checks.append(checks)
    print(checks)

passed = sum(
    check["structure_pass"]
    for check in product_final_structure_checks
)

print(f"\nStructure pass rate: {passed}/15 = {100 * passed / 15:.1f}%")



--- Run 1 ---
Wireless Mouse  
This wireless mouse is priced at $29.99. The information provided indicates that it operates without a connection cable during use. For more details regarding features or specifications, please refer to the product information available.

--- Run 2 ---
Wireless Mouse  
This wireless mouse is available for $29.99. It operates without a connection cable, allowing for a cable-free experience during use. For more information regarding specifications and features, please refer to the product details provided.

--- Run 3 ---
Wireless Mouse  
This wireless mouse is offered at a price of $29.99. It operates without a connection cable to the source device during use. For more information regarding its specifications or features, please refer to additional product details.

--- Run 4 ---
Wireless Mouse  
This wireless mouse is available for $29.99. The product is designed for cable-free operation, eliminating the need for a connection cable during use. For further

In [47]:
from datetime import datetime, timezone
from pathlib import Path
import json

# Collect only the lab's prompts, outputs, and evaluation data.
experiment_data = {
    name: value
    for name, value in list(globals().items())
    if name.startswith(("sentiment_", "product_", "extraction_"))
    and isinstance(value, (str, list, dict))
}

export_data = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "configuration": {
        "model": MODEL,
        "temperature": TEMPERATURE,
        "max_completion_tokens": 500,
    },
    "experiments": experiment_data,
    "call_history": call_history,
}

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
export_path = results_dir / f"prompt_experiments_{timestamp}.json"

with export_path.open("x", encoding="utf-8") as file:
    json.dump(export_data, file, ensure_ascii=False, indent=2)

print("Saved:", export_path)
print("API responses recorded:", len(call_history))
print("Experiment variables saved:", len(experiment_data))

Saved: results/prompt_experiments_20261014T145106783436Z.json
API responses recorded: 187
Experiment variables saved: 46


## Final Evaluation and Failure Analysis

### Evaluation approach

Consistency means adherence to defined task requirements, not identical
wording. Format compliance and factual correctness were assessed separately.

- Sentiment: exact allowed label and agreement with the reference label.
- Product: required structure, length, price, and manual grounding review.
- Extraction: JSON validity, schema compliance, and reference-value correctness.

### Results

| Task and metric | v1 — 15 runs | v2 — 5 runs | Final v3 — 15 runs |
|---|---:|---:|---:|
| Sentiment: correct standard label | 0% | 100% | 100% |
| Product: unsupported-claim-free outputs | 0% | 60% (v2b) | 100% |
| Extraction: valid JSON | 0% | 100% | 100% |
| Extraction: correct schema and values | 0% | 100% | 100% |

Product v2b grounding was assessed manually: runs 1 and 5 failed.
The earlier product v3 achieved 80% combined compliance.
The final revised v3 achieved 100% on the original product input.

For extraction v1, 4/15 responses preserved the item identifier meaning,
but none satisfied the target JSON contract. Its 0% combined score
does not mean that every extracted detail was incorrect.

### Observed failure patterns

- Sentiment v1 conveyed positive sentiment correctly, but returned
  explanations, alternative labels, and varying Markdown.
- Product v1 invented names, battery specifications, connectivity,
  compatibility, and other unsupported features.
- Product v2 reduced fabricated specifications but retained unsupported
  performance claims and exaggerated cable-free benefits.
- Extraction v1 frequently confused item identifiers with order identifiers
  and used inconsistent field names.

### Improvement techniques

- Explicit labels and output constraints resolved the observed sentiment
  formatting failures.
- Schema definitions, missing-value rules, and identifier distinctions
  improved extraction.
- Few-shot examples and iterative grounding constraints improved product
  descriptions. An ambiguity example helped avoid guessing the meaning
  of "Notebook".
- Extraction v3 used an internal step-by-step checking instruction while
  returning only JSON. Output quality does not prove that the model
  followed those internal steps.

### Task variations

- Sentiment: 4/4 new cases passed, including mixed sentiment and sarcasm.
- Extraction: 3/3 cases passed, covering missing information, an order-only
  identifier, and separate item and order identifiers.
- Product variations exposed unsupported performance claims, word-count
  failures, and ambiguity errors. These cases were then used for refinement.
- After adding an ambiguity example, one Notebook test passed with 38 words.
  The complete product variation set was not retested on that final revision.

### Limitations and trade-offs

- Repeated calls on one input measure repeatability, not broad generalization.
- Five-run and fifteen-run percentages have different sample sizes.
- Variation cases used for refinement are development cases, not held-out tests.
- Product grounding involved manual judgment.
- Final product descriptions became more factual but less persuasive;
  persuasiveness was not scored.
- Few-shot and step-by-step instructions showed no additional gain over v2
  on the already-successful sentiment and extraction baseline inputs.
- These results support a classroom PoC, not production readiness.